In [2]:
import os
import numpy as np
import cv2
import librosa
import soundfile as sf
from tqdm import tqdm

# ============ NEW (Week 1): frequency remapping — copied unchanged from Img2Audv3 ============
def frequency_remap(spectrogram, sr=22050, n_fft=1024, target_low=1000.0, target_high=4000.0):
    """
    Warps a linear-frequency magnitude spectrogram so ALL of its structural
    content is compressed into the [target_low, target_high] Hz band.

    f_new = a * log(f_old + 1) + b   maps [0, sr/2] -> [target_low, target_high]

    We need the INVERSE (a "pull"/backward mapping): for every output bin
    whose true physical frequency falls in the target band, ask "which
    original frequency warped TO here?" and interpolate that value in.
    Bins outside the band are left at zero (silence).
    """
    n_bins, n_frames = spectrogram.shape
    freq_axis = np.linspace(0, sr / 2, n_bins)  # true Hz of each bin, 0..11025

    b = target_low
    a = (target_high - target_low) / np.log(freq_axis[-1] + 1)

    remapped = np.zeros_like(spectrogram)
    in_band = (freq_axis >= target_low) & (freq_axis <= target_high)
    f_true = freq_axis[in_band]

    f_source = np.exp((f_true - b) / a) - 1.0
    src_bin_pos = np.clip(f_source / (sr / n_fft), 0, n_bins - 1)

    src_bins = np.arange(n_bins)
    for t in range(n_frames):
        remapped[in_band, t] = np.interp(src_bin_pos, src_bins, spectrogram[:, t])

    return remapped, a, b
# ============ END NEW ============


def image_to_audio_dataset(input_root, output_root, sr=22050, apply_remap=True):  # NEW: apply_remap flag (Week 1)
    """
    Convert all images in a dataset to audio files with verbose debugging
    """
    # Spectrogram parameters
    n_fft = 1024
    hop_length = 512
    target_height = n_fft // 2 + 1
    target_width = 128
    
    # Create output root if it doesn't exist
    os.makedirs(output_root, exist_ok=True)
    print(f"Output root created at: {output_root}")
    
    # ============ NEW (Week 1): report remap params once, up front ============
    # a and b only depend on sr / n_fft / target_height (all fixed for this run),
    # never on pixel content -- so they're identical for every image. Compute them
    # once here for a log line instead of inside the loop (would be 9000+ identical prints).
    if apply_remap:
        _, remap_a, remap_b = frequency_remap(np.zeros((target_height, 1)), sr=sr, n_fft=n_fft)
        print(f"Frequency remap ON -> a={remap_a:.2f}, b={remap_b:.2f} (targeting 1-4kHz band)")
    else:
        print("Frequency remap OFF (this reproduces the original baseline I-GLA audio)")
    # ============ END NEW ============
    
    # Counters for stats
    total_files = 0
    processed_files = 0
    skipped_files = 0
    error_files = 0
    
    # Process all images
    for root, dirs, files in os.walk(input_root):
        for file in tqdm(files, desc=f"Processing {os.path.basename(root)}"):
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                total_files += 1
                try:
                    # Create mirror directory structure
                    relative_path = os.path.relpath(root, input_root)
                    output_dir = os.path.join(output_root, relative_path)
                    os.makedirs(output_dir, exist_ok=True)
                    
                    # Set paths
                    input_path = os.path.join(root, file)
                    output_path = os.path.join(output_dir, f"{os.path.splitext(file)[0]}.wav")
                    
                    # Skip existing files
                    if os.path.exists(output_path):
                        skipped_files += 1
                        continue
                    
                    # Load image
                    image = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
                    if image is None:
                        print(f"\nWarning: Could not read image {input_path}")
                        error_files += 1
                        continue
                        
                    # Process image
                    image = cv2.resize(image, (target_width, target_height))
                    image_normalized = np.flipud(image.astype(np.float32) / 255.0)
                    
                    # Convert to audio
                    spectrogram_db = librosa.amplitude_to_db(image_normalized + 1e-7, ref=np.max)
                    spectrogram_linear = librosa.db_to_amplitude(spectrogram_db)
                    
                    # ============ NEW (Week 1): apply frequency remap here ============
                    if apply_remap:
                        spectrogram_linear, _, _ = frequency_remap(spectrogram_linear, sr=sr, n_fft=n_fft)
                    # ============ END NEW ============
                    
                    audio = librosa.griffinlim(
                        spectrogram_linear,
                        n_iter=32,
                        n_fft=n_fft,
                        hop_length=hop_length,
                        win_length=n_fft
                    )
                    
                    # Save audio
                    sf.write(output_path, audio, sr)
                    processed_files += 1
                    
                except Exception as e:
                    print(f"\nError processing {file}: {str(e)}")
                    error_files += 1
                    continue
    
    # Print summary
    print("\nConversion Summary:")
    print(f"Total image files found: {total_files}")
    print(f"Successfully processed: {processed_files}")
    print(f"Skipped (already existed): {skipped_files}")
    print(f"Errors encountered: {error_files}")

# Example usage - VERIFY THESE PATHS!
input_dataset_root = r"D:\Documents\Iquisitionis\103\data\vehicle_spectrogram"
# NEW: separate output folder so the original baseline audio (sportsball-aud) isn't
# overwritten -- Week 1's plan needs BOTH old and new audio to compare CNN accuracy.
output_dataset_root = r"D:\Documents\Iquisitionis\103v2\data\4Class_remapped"

# Verify input path exists
if not os.path.exists(input_dataset_root):
    print(f"ERROR: Input path does not exist: {input_dataset_root}")
else:
    print(f"Input directory verified: {input_dataset_root}")
    image_to_audio_dataset(input_dataset_root, output_dataset_root, apply_remap=True)
    print(f"\nCheck output directory: {output_dataset_root}")

Input directory verified: D:\Documents\Iquisitionis\103\data\vehicle_spectrogram
Output root created at: D:\Documents\Iquisitionis\103v2\data\4Class_remapped
Frequency remap ON -> a=322.30, b=1000.00 (targeting 1-4kHz band)


Processing vehicle_spectrogram: 0it [00:00, ?it/s]
Processing test: 0it [00:00, ?it/s]
Processing Truck: 100%|████████████████████████████████████████████████████████████████| 20/20 [00:08<00:00,  2.48it/s]
Processing train: 0it [00:00, ?it/s]
Processing Truck: 100%|████████████████████████████████████████████████████████████████| 80/80 [00:31<00:00,  2.52it/s]


Conversion Summary:
Total image files found: 403
Successfully processed: 403
Skipped (already existed): 0
Errors encountered: 0

Check output directory: D:\Documents\Iquisitionis\103v2\data\4Class_remapped
